In [ ]:
%load_ext autoreload
%autoreload 2

import os

os.environ["CUDA_VISIBLE_DEVICES"] = "2"

print(os.getcwd())
project_root = os.getcwd()
while not os.path.exists(os.path.join(project_root, "pyproject.toml")) and project_root != os.path.dirname(project_root):
    project_root = os.path.dirname(project_root)
os.chdir(project_root)
print(os.getcwd())

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick 
import numpy as np
from model_utils.model_config import get_model_path
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
# model_name_list = ["qwen3-4b", "qwen3-8b", "qwen3-14b" ,"toolace-2.5-8b", "watt-tool-8b"]
model_name_list = ["qwen3-4b", "qwen3-8b", "qwen3-14b" ,"toolace-2.5-8b",  "watt-tool-8b"]
model_dsiplay_name_dict = {
    "qwen3-4b": "Qwen3-4B",
    "qwen3-8b": "Qwen3-8B",
    "qwen3-14b": "Qwen3-14B",
    "toolace-2.5-8b": "ToolACE-2.5-8B",
    "watt-tool-8b": "Watt-Tool-8B"
} 


add_param_num_list=[0, 1, 2, 3, 4]

In [ ]:
result = {}
for model_name in model_name_list:
    result[model_dsiplay_name_dict[model_name]] = {}

    file_dir = os.path.join("data", model_name)

    if model_name in ["toolace-2.5-8b", "watt-tool-8b"]:
        tokenizer = AutoTokenizer.from_pretrained(get_model_path(model_name))

    def check_tool_call(item):
        if model_name not in ["toolace-2.5-8b", "watt-tool-8b"]:
            return 1 if item["logits_info"]["tool_call_token_rank"]==0 else 0
        else:
            out_str = tokenizer.decode(item["logits_info"]["top_token_ids"][0], skip_special_tokens=True)
            if "]" not in out_str:
                if out_str.startswith("["):
                    if len(out_str) == 1:
                        return 1
                    elif out_str[1] == item["tool_name"][0]:
                        return 1
            return 0


    for add_param_num in add_param_num_list:

        pair_file_path = os.path.join(file_dir, f"pair_add_{add_param_num}.json")
        with open(pair_file_path, "r", encoding="utf-8") as f:
            pair_list = json.load(f)
        pair_list = [check_tool_call(item) for item in pair_list]

        result[model_dsiplay_name_dict[model_name]][add_param_num] = pair_list

In [ ]:
np.mean(result["Watt-Tool-8B"][1])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MaxNLocator

plt.rcParams['text.usetex'] = False 

# Use Computer Modern for math text

plt.rcParams['mathtext.fontset'] = 'cm'


plt.rcParams['font.family'] = 'serif'

def plot_line_chart_custom_margin(result_data, margin_ratio=0.1):
    """
    Plot per-model TIR across dataset subsets (D_0..D_4).
    
    Args:
    result_data: nested dict {model: {k: [values...]}}
    margin_ratio: vertical padding fraction (default 0.15).
                  
                  
    """
    
    
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica']
    plt.rcParams['font.size'] = 12
    plt.rcParams['axes.linewidth'] = 1.2
    
    
    markers = ['o', 's', '^', 'D', 'v', 'p', '*'] 
    colors = ['#A1A9D0', '#F0988C', '#B883D4', '#9E9E9E', '#96CCCB', '#8c564b']

    
    fig, ax = plt.subplots(figsize=(6, 3), dpi=300)
    
    model_names = list(result_data.keys())
    
    
    global_max_y = -np.inf
    global_min_y = np.inf
    
    
    all_x_ticks = set()

    # Plot each model
    for i, model_name in enumerate(model_names):
        param_dict = result_data[model_name]
        
        plot_points = []
        
        for param_key, values in param_dict.items():
            
            try:
                x_val = int(param_key)
            except ValueError:
                x_val = float(param_key)
            
            all_x_ticks.add(x_val) 
            
            if len(values) > 0:
                mean_val = np.mean(values) * 100
                err_val = (np.std(values, ddof=1) / np.sqrt(len(values))) * 100
            else:
                mean_val, err_val = 0, 0
            
            plot_points.append((x_val, mean_val, err_val))
        
        
        plot_points.sort(key=lambda x: x[0])
        
        if not plot_points:
            continue
            
        xs = [p[0] for p in plot_points]
        ys = [p[1] for p in plot_points]
        errs = [p[2] for p in plot_points]
        
        
        curr_max = max([y + e for y, e in zip(ys, errs)])
        curr_min = min([y - e for y, e in zip(ys, errs)])
        
        if curr_max > global_max_y: global_max_y = curr_max
        if curr_min < global_min_y: global_min_y = curr_min
        
        
        ax.errorbar(xs, ys, yerr=errs, label=model_name,
                    marker=markers[i % len(markers)], 
                    color=colors[i % len(colors)],
                    linewidth=1.5, markersize=7,
                    mec='black',       
                    markeredgewidth=0.8,
                    capsize=0, elinewidth=1.2) 

    # Axes

    
    sorted_ticks = sorted(list(all_x_ticks))
    ax.set_xticks(sorted_ticks) 
    ax.set_xticklabels(sorted_ticks, fontsize=13) 
    
    
    
    y_range = global_max_y - global_min_y
    if y_range == 0: y_range = 10 
    
    
    
    
    bottom_limit = max(0, global_min_y - (y_range * margin_ratio))
    top_limit = min(105, global_max_y + (y_range * margin_ratio)) 
    
    ax.set_ylim(bottom_limit, top_limit)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=10))

    # Legend
    ax.set_xlabel(r'Number of Added Attribute-Parameter ($\mathcal{D}_{0},\dots,\mathcal{D}_{4}$)', fontsize=13, fontweight='bold')
    ax.set_ylabel('Tool Invocation Rate (%)', fontsize=13, fontweight='bold')
    
    # ax.legend(loc='best', frameon=False, fontsize=10)
    ax.legend(
        loc='lower center',
        bbox_to_anchor=(0.5, 1.0),
        ncol=5,                   
        frameon=False,            
        fontsize=8.5,
        handletextpad=0.3,        
        columnspacing=0.7         
    )
    ax.grid(True, linestyle='--', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig("figs/behavior/alignment_degree.pdf", bbox_inches="tight")
    plt.show()

plot_line_chart_custom_margin(result)